## Zadania domowe - Seria 5

Napisać iterator, który generuje kolejne dzielniki liczby naturalnej $N$.
Wypisać dzielniki wybranej liczby $N$ korzystając z pętli for i iteratora.
Utworzyć listę dzielników wybranej liczby $N$ korzystając z list comprehensions.

In [180]:
from builtins import StopIteration


class DivisorIterator:
    def __init__(self, n):
        self._n = n
        self._actual = 1
        
    def __iter__(self):
        return self
    
    def __next__(self):
        while self._actual <= self._n:
            if self._n % self._actual == 0:
                ret = self._actual
                self._actual += 1
                return ret
            else:
                self._actual += 1
        raise StopIteration
N = 28
di = DivisorIterator(N)

for divisor in di:
    print(divisor , end=" ")
    
print()
l = [elem for elem in DivisorIterator(N)]
print(l)

1 2 4 7 14 28 
[1, 2, 4, 7, 14, 28]


Napisać iterator, który generuje pary (dzielnik pierwszy, krotność dzielnika) dla liczby $N$.
Wypisać dzielniki pierwsze wybranej liczby $N$ wraz z krotnościami korzystając z pętli for i iteratora.
Utworzyć słownik o kluczach będących dzielnikami pierwszymi wybranej liczby $N$
i wartościach będących krotnościami odpowiednich dzielników korzystając z dictionary comprehensions.

In [181]:
class PrimeDivisorIterator:
    def __init__(self, n):
        self.n = n
        self.current_divisor = 2
        self.count = 0

    def __iter__(self):
        return self

    def __next__(self):
        while self.n > 1:
            if self.n % self.current_divisor == 0:
                self.count += 1
                self.n //= self.current_divisor
                if self.n % self.current_divisor != 0:
                    return self.current_divisor, self.count 
            else:
                self.count = 0
                self.current_divisor += 1
        raise StopIteration

N = 60
pdi = PrimeDivisorIterator(N)
for divisor, count in pdi:
    print(divisor, "-", count, end=", ")
print()
pd = {divisor : count for divisor , count in PrimeDivisorIterator(N)}
print(pd)


2 - 2, 3 - 1, 5 - 1, 
{2: 2, 3: 1, 5: 1}


Dla klasy `Poly` z poprzedniej serii zadań dodać protokół iteratora.
Iterator powinien zwracać kolejne podwielomiany (również klasy `Poly`) będące składnikami wielomianu
począwszy od wyrazu wolnego. Na przykład dla wielomianu
$$
P(x) = 3x^5 + 2x^3 + 5x + 1
$$
Iterator powinien zwracać kolejno wielomiany
$$
P_1(x)=1,\ P_2(x)=5x,\ P_3(x)=2x^3,\ P_4(x)=3x^5
$$
Utworzyć wielomian 6-tego stopnia o współczynnikach całkowitych losowanych z przedziału $[-5,5]$.
Przetestować działanie iteratora obliczając sumę wartości kolejno generowanych podwielomianów
dla wybranej liczby $x$. Porównać zsumowaną wartość z wartością oryginalnego wielomianu dla zmiennej $x$.



In [269]:
import random

class Poly:
    def __init__(self, coefficients):
        self.coefficients = coefficients

    def __str__(self):
        result = ""
        degree = len(self.coefficients) - 1
        for coef in self.coefficients:
            if coef != 0:
                if degree > 1:
                    result += f"{coef}x^{degree} + "
                elif degree == 1:
                    result += f"{coef}x + "
                else:
                    result += f"{coef}"
            degree -= 1
        return result.rstrip(" + ")

    def __eq__(self, other):
        return self.coefficients == other.coefficients

    def __ne__(self, other):
        return not self.__eq__(other)

    def __add__(self, other):
        max_length = max(len(self.coefficients), len(other.coefficients))
        result = [0] * max_length
        for i in range(len(self.coefficients)):
            result[i] += self.coefficients[i]
        for i in range(len(other.coefficients)):
            result[i] += other.coefficients[i]
        return Poly(result)

    def __sub__(self, other):
        max_length = max(len(self.coefficients), len(other.coefficients))
        result = [0] * max_length
        for i in range(len(self.coefficients)):
            result[i] += self.coefficients[i]
        for i in range(len(other.coefficients)):
            result[i] -= other.coefficients[i]
        return Poly(result)

    def __neg__(self):
        return Poly([-coef for coef in self.coefficients])

    def __mul__(self, other):
        if type(other) == Poly:
            result = [0] * (len(self.coefficients) + len(other.coefficients) - 1)
            for i in range(len(self.coefficients)):
                for j in range(len(other.coefficients)):
                    result[i + j] += self.coefficients[i] * other.coefficients[j]
            return Poly(result)
        else: 
            return Poly([coef * other for coef in self.coefficients])
        

    def __rmul__(self, other):
        return Poly([other * coef for coef in self.coefficients])

    def degree(self):
        return len(self.coefficients) - 1

    def __call__(self, x):
        result = 0
        max_degree = self.degree()
        for i in range(len(self.coefficients)):
            result += self.coefficients[i] * (x ** max_degree)
            max_degree -= 1
        return result

    def __iter__(self):
        return PolyIterator(self)

class PolyIterator:
    def __init__(self, poly):
        self.poly = poly
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.index >= len(self.poly.coefficients):
            raise StopIteration
        else:
            coeffs = [0] * len(self.poly.coefficients)
            coeffs[len(self.poly.coefficients) - 1 - self.index] = self.poly.coefficients[self.index]
            self.index += 1
            return Poly(coeffs)

# Tworzenie wielomianu 6-tego stopnia o współczynnikach całkowitych losowanych z przedziału [-5, 5]
coefficients = [random.randint(-5, 5) for _ in range(7)]
p = Poly(coefficients)
print(p)

x = 2
sum_of_values = 0
for poly in p:
        value = poly(x)
        print(f"P_{len(poly.coefficients)}(x) =", poly, ", Value =", value)
        sum_of_values += value

print("Sum of values from sub-polynomials:", sum_of_values)
print("Value of original polynomial for x =", x, ":", p(x))


5x^6 + -1x^5 + 3x^4 + 5x^2 + -5x + 3
P_7(x) = 5 , Value = 5
P_7(x) = -1x , Value = -2
P_7(x) = 3x^2 , Value = 12
P_7(x) =  , Value = 0
P_7(x) = 5x^4 , Value = 80
P_7(x) = -5x^5 , Value = -160
P_7(x) = 3x^6 , Value = 192
Sum of values from sub-polynomials: 127
Value of original polynomial for x = 2 : 349


Dla klasy `Poly` z poprzedniej serii zadań dodać protokół sekwencji.
Operator indeksowania powinien zwracać odpowiedni podwielomian (również klasy `Poly`)
odpowiadający składnikowi wielomianu stopnia równego indeksowi. Na przykład dla wielomianu
$$
P(x) = 3x^5 + 2x^3 + 5x + 1
$$
oprerator indeksu powinnien zwracać wielomiany
$$
P[0]=1, \\
P[1]=5x,\\
P[2]=2x^3,\\
P[3]=3x^5
$$
Utworzyć wielomian 6-tego stopnia o współczynnikach całkowitych losowanych z przedziału $[-5,5]$
i przetestować działanie operatora indeksu.



In [155]:
import random

class Poly:
    def __init__(self, coefficients):
        self.coefficients = coefficients

    def __str__(self):
        result = ""
        degree = len(self.coefficients) - 1
        for coef in self.coefficients:
            if coef != 0:
                if degree > 1:
                    result += f"{coef}x^{degree} + "
                elif degree == 1:
                    result += f"{coef}x + "
                else:
                    result += f"{coef}"
            degree -= 1
        return result.rstrip(" + ")

    def degree(self):
        return len(self.coefficients) - 1

    def sub_poly(self, index):
        if index < 0 or index > self.degree():
            raise IndexError("Index out of range")
        return Poly([0] * index + [self.coefficients[index]])

    def index(self, index):
        return self.sub_poly(index)

print("")
# Tworzenie wielomianu 6-tego stopnia o współczynnikach całkowitych losowanych z przedziału [-5,5]
coefficients = [random.randint(-5, 5) for _ in range(7)]
P = Poly(coefficients)
print("Wielomian P(x):", P)

# Testowanie działania operatora indeksu
print("P.index(0) =", P.index(0))
print("P.index(1) =", P.index(1))
print("P.index(2) =", P.index(2))
print("P.index(3) =", P.index(3))



Wielomian P(x): -2x^6 + 3x^5 + -2x^4 + 1x^3 + -2x^2 + 2x + -2
P.index(0) = -2
P.index(1) = 3
P.index(2) = -2
P.index(3) = 1


Okazuje się, że protokół iteracji można wywołać niejawnie jeżeli klasa ma zaimplementowany protokół sekwencji.
Zamiast metody `__next__()` wywoływana jest metoda `__getitem__()` dla elementów o kolejnych indeksach.
Jednak w takim przypadku operator indeksowania powinien rzucać wyjątek `IndexError` jeżeli indeks
znajduje się poza ustalownym zakresem.
Zmodyfikować metodę `__getitem__()` dla klasy z poprzedniego zadania.
Przetestować działanie iteratora obliczając sumę wartości kolejno generowanych podwielomianów
dla wybranej liczby $x$. Porównać zsumowaną wartość z wartością oryginalnego wielomianu dla zmiennej $x$.

Wielomian P(x): 4x^6 + 4x^5 + 2x^4 + -4x^3 + -2x^2 + -4x + 3


TypeError: 'int' object is not callable